# 29. End-to-end final discharge and regression

This notebook is the stage-90 **live final-discharge harness**.

Its job is different from the earlier notebooks:

1. run the **full live theorem-program discharge route** through `proof_driver.build_golden_theorem_program_discharge_status_report(...)`;
2. record the exact **acceptance criteria** for a genuine end-to-end closure claim;
3. expose the **Theorem VIII burden fields** and reduction-geometry diagnostics that determine what still remains;
4. optionally launch the key **pytest regression tiers** from inside the notebook.

This notebook is intended to answer the question:

> “If I run the repository on its own default theorem route, does the project now close live, and if not, what exact burden remains?”

## Important distinction

The stage-90 regression tests include strong theorem-object and reduction tests, but some of them still verify
the endgame by feeding in already-final certificates or monkeypatching the top-level driver. That is useful
regression coverage, but it is **not** the same as a fully live default-path discharge.

This notebook is meant to be the **live counterpart** to those staged tests.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = None
for cand in [Path.cwd(), Path.cwd().parent, *Path.cwd().parents]:
    if (cand / "kam_theorem_suite").exists():
        PROJECT_ROOT = cand
        break
if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root containing kam_theorem_suite")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = /global/u2/s/suryact/Theory/KAM/proof20/vclose


In [2]:
import json
import os
import platform
import shlex
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
from pprint import pprint

try:
    import pandas as pd
except Exception:
    pd = None

from kam_theorem_suite import proof_driver
from kam_theorem_suite.standard_map import HarmonicFamily
from kam_theorem_suite.golden_supercritical import generate_golden_convergent_specs
from kam_theorem_suite.adaptive_incompatibility import (
    build_adaptive_incompatibility_entry_certificate,
    build_adaptive_incompatibility_atlas_certificate_from_entries,
    build_golden_adaptive_incompatibility_certificate_from_atlas,
)
from kam_theorem_suite.adaptive_tail_coherence import (
    build_golden_adaptive_tail_coherence_entry,
    build_golden_adaptive_tail_coherence_certificate_from_entries,
)
from kam_theorem_suite.theorem_iv_tail_transport import (
    build_golden_tail_band_transport_certificate,
    make_tail_transport_entry_dicts,
)


## Configuration

The defaults below are chosen to make the notebook usable as the canonical stage-90 live-discharge route.

- `BASE_K_VALUES` uses the same small canonical pair used repeatedly in the late-stage tests.
- `RUN_LIVE_DISCHARGE` should usually stay `True`.
- The regression tiers are split so you can run them gradually:
  - **tier 1:** stage-84 through stage-90 theorem-program tests;
  - **tier 2:** focused endgame/driver tests;
  - **tier 3:** broader active-tree pytest run with the archival `stage35/` subtree excluded.

Set `FAIL_HARD_ON_ACCEPTANCE` to `True` if you want the notebook to stop immediately when the live discharge does not meet your closure target.

In [3]:
BASE_K_VALUES = [0.20, 0.25]
FAMILY = HarmonicFamily()
CHALLENGER_SPECS = None

LIVE_DRIVER_KWARGS = {
    # Add any explicit theorem-driver overrides here if you decide to standardize them later.
    # Example:
    # "statement_mode": "one-variable",
}

RUN_LIVE_DISCHARGE = True
FAIL_HARD_ON_ACCEPTANCE = False

# The optimized driver now stages shared certificates internally.
# In addition, the notebook can cache each stage to disk so interrupted runs
# can resume without rebuilding everything from scratch.
USE_STAGE_CACHE = True
FORCE_REBUILD_STAGE_CACHE = False

RUN_REGRESSION_TIER_1 = False   # stage84-stage90 theorem-program slice
RUN_REGRESSION_TIER_2 = False   # focused endgame / driver / reduction slice
RUN_REGRESSION_TIER_3 = False   # broad active-tree pytest run excluding archival stage35/

PYTEST_QUIET = True
PYTEST_MAXFAIL = None           # e.g. 1 for a quick stop-on-first-failure pass
PYTEST_EXTRA_ARGS = []          # e.g. ["-k", "stage90"]

ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "final_discharge"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
STAGE_CACHE_DIR = ARTIFACT_DIR / "stage_cache"
STAGE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Theorem IV upper-side decomposition controls
THEOREM_IV_UPPER_ENTRY_N_JOBS = max(1, min(4, os.cpu_count() or 1))
THEOREM_IV_UPPER_ENTRY_PARALLELISM = "serial"


In [4]:
import json
from pathlib import Path
import numpy as np


def _display_records(records, *, title=None):
    if title:
        print(title)
    if pd is not None:
        return pd.DataFrame(records)
    pprint(records)
    return records

def _jsonify(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonify(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonify(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return _jsonify(obj.tolist())
    if isinstance(obj, (np.integer, np.floating, np.bool_)):
        return obj.item()
    if isinstance(obj, Path):
        return str(obj)
    if hasattr(obj, "to_dict"):
        try:
            return _jsonify(obj.to_dict())
        except Exception:
            pass
    if hasattr(obj, "tolist") and not isinstance(obj, (str, bytes)):
        try:
            return _jsonify(obj.tolist())
        except Exception:
            pass
    return obj


def _json_dump(path: Path, obj) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(_jsonify(obj), indent=2, sort_keys=True),
        encoding="utf-8",
    )
    print(f"Wrote {path}")
    return path

def _theorem_rows_table(report: dict) -> list[dict]:
    rows = []
    for row in report.get("theorem_status_rows", []):
        rows.append({
            "theorem_name": row.get("theorem_name"),
            "theorem_status": row.get("theorem_status"),
            "statement_mode": row.get("statement_mode"),
            "open_hypotheses_count": len(row.get("open_hypotheses", []) or []),
            "active_assumptions_count": len(row.get("active_assumptions", []) or []),
        })
    return rows


def _build_acceptance_snapshot(report: dict) -> dict:
    summary = report.get("implementation_summary", {})
    theorem_viii = (report.get("subreports") or {}).get("theorem_viii", {})
    geometry = report.get("current_reduction_geometry_summary", {})

    return {
        "overall_theorem_status": summary.get("overall_theorem_status"),
        "theorem_viii_final_status": summary.get("theorem_viii_final_status"),
        "current_reduction_geometry_available": bool(summary.get("current_reduction_geometry_available", False)),
        "current_reduction_geometry_status": summary.get("current_reduction_geometry_status"),
        "current_reduction_geometry_minimum_certified_margin": summary.get("current_reduction_geometry_minimum_certified_margin"),
        "final_universal_theorem_ready_for_code_path": bool(summary.get("final_universal_theorem_ready_for_code_path", False)),
        "final_universal_theorem_ready_for_paper": bool(summary.get("final_universal_theorem_ready_for_paper", False)),
        "true_mathematical_burden_remaining": list(summary.get("true_mathematical_burden_remaining", []) or []),
        "paper_grade_burden_remaining": list(summary.get("paper_grade_burden_remaining", []) or []),
        "workstream_residual_caveat": list(summary.get("workstream_residual_caveat", []) or []),
        "recommended_next_move_kind": summary.get("recommended_next_move_kind"),
        "recommended_next_move_target": summary.get("recommended_next_move_target"),
        "recommended_next_move_action": summary.get("recommended_next_move_action"),
        "recommended_next_move_rationale": summary.get("recommended_next_move_rationale"),
        "theorem_viii_remaining_true_mathematical_burden": list(theorem_viii.get("remaining_true_mathematical_burden", []) or []),
        "theorem_viii_remaining_workstream_paper_grade_burden": list(theorem_viii.get("remaining_workstream_paper_grade_burden", []) or []),
        "theorem_viii_remaining_exhaustion_paper_grade_burden": list(theorem_viii.get("remaining_exhaustion_paper_grade_burden", []) or []),
        "geometry_status": geometry.get("status"),
        "geometry_source": geometry.get("source"),
        "geometry_pending_count": geometry.get("pending_count"),
        "geometry_minimum_certified_margin": geometry.get("minimum_certified_margin"),
    }


def _acceptance_checks(snapshot: dict) -> list[dict]:
    checks = [
        {
            "check": "theorem_viii final status exists",
            "passed": bool(snapshot.get("theorem_viii_final_status")),
            "value": snapshot.get("theorem_viii_final_status"),
        },
        {
            "check": "current reduction geometry available",
            "passed": bool(snapshot.get("current_reduction_geometry_available")),
            "value": snapshot.get("current_reduction_geometry_available"),
        },
        {
            "check": "current reduction geometry strong",
            "passed": snapshot.get("current_reduction_geometry_status") == "current-reduction-geometry-strong",
            "value": snapshot.get("current_reduction_geometry_status"),
        },
        {
            "check": "no true mathematical burden remains",
            "passed": len(snapshot.get("true_mathematical_burden_remaining", [])) == 0,
            "value": snapshot.get("true_mathematical_burden_remaining"),
        },
        {
            "check": "code-path final certificate ready",
            "passed": bool(snapshot.get("final_universal_theorem_ready_for_code_path", False)),
            "value": snapshot.get("final_universal_theorem_ready_for_code_path"),
        },
        {
            "check": "paper-grade final certificate ready",
            "passed": bool(snapshot.get("final_universal_theorem_ready_for_paper", False)),
            "value": snapshot.get("final_universal_theorem_ready_for_paper"),
        },
        {
            "check": "no paper-grade burden remains",
            "passed": len(snapshot.get("paper_grade_burden_remaining", [])) == 0,
            "value": snapshot.get("paper_grade_burden_remaining"),
        },
    ]
    return checks


def _summarize_next_step(snapshot: dict) -> dict:
    true_burden = list(snapshot.get("true_mathematical_burden_remaining", []) or [])
    paper_burden = list(snapshot.get("paper_grade_burden_remaining", []) or [])
    ready_code = bool(snapshot.get("final_universal_theorem_ready_for_code_path", False))
    ready_paper = bool(snapshot.get("final_universal_theorem_ready_for_paper", False))

    if ready_code and ready_paper and not true_burden and not paper_burden:
        diagnosis = "full-closure-signal"
        recommendation = "The live default-path discharge appears to close both code-path and paper-grade finality. Next steps are repo hygiene, broad regression, and external writeup."
    elif ready_code and not ready_paper and not true_burden:
        diagnosis = "paper-grade-packaging-burden-only"
        recommendation = "The mathematics appears closed on the live route, but paper-grade burdens remain. Focus next on Workstream-A / Theorem VII cleanup and citation/finality packaging."
    elif not ready_code and true_burden:
        diagnosis = "live-mathematical-burden-remains"
        recommendation = "The live route is still exposing true mathematical burdens. Use the VIII burden lists below as the canonical guide for the next theorem-level move."
    else:
        diagnosis = "mixed-endgame-state"
        recommendation = "The project is in a mixed endgame state. Inspect the VIII burden lists, the geometry summary, and the theorem status rows before deciding whether the next move is mathematical or packaging."
    return {
        "diagnosis": diagnosis,
        "recommendation": recommendation,
    }


def _run_command(cmd, *, cwd=PROJECT_ROOT):
    pretty = " ".join(shlex.quote(str(x)) for x in cmd)
    print(f"$ {pretty}")
    started = time.time()
    proc = subprocess.run(
        [str(x) for x in cmd],
        cwd=str(cwd),
        capture_output=True,
        text=True,
    )
    elapsed = time.time() - started
    print(f"[exit={proc.returncode}] [elapsed={elapsed:.2f}s]")
    if proc.stdout:
        print("\n--- stdout ---\n")
        print(proc.stdout)
    if proc.stderr:
        print("\n--- stderr ---\n")
        print(proc.stderr)
    return {
        "cmd": [str(x) for x in cmd],
        "returncode": int(proc.returncode),
        "elapsed_seconds": elapsed,
        "stdout": proc.stdout,
        "stderr": proc.stderr,
    }


def _pytest_cmd(paths_or_targets):
    cmd = [sys.executable, "-m", "pytest"]
    if PYTEST_QUIET:
        cmd.append("-q")
    if PYTEST_MAXFAIL is not None:
        cmd.extend(["--maxfail", str(PYTEST_MAXFAIL)])
    cmd.extend(["--ignore=stage35"])
    cmd.extend(PYTEST_EXTRA_ARGS)
    cmd.extend(list(paths_or_targets))
    return cmd


def _collect_stage84_to_stage90_tests():
    tests_dir = PROJECT_ROOT / "tests"
    targets = sorted(str(p.relative_to(PROJECT_ROOT)) for p in tests_dir.glob("test_stage8*.py"))
    targets += sorted(str(p.relative_to(PROJECT_ROOT)) for p in tests_dir.glob("test_stage90*.py"))
    return targets


def _focused_endgame_tests():
    return [
        "tests/test_stage89_final_universal_reduction.py",
        "tests/test_stage89_proof_driver_endgame_summary.py",
        "tests/test_stage90_workstream_final_certificate.py",
        "tests/test_stage90_universality_class_finalization.py",
        "tests/test_stage90_renormalization_package_finalization.py",
        "tests/test_stage90_critical_surface_finalization.py",
        "tests/test_stage90_viii_paper_finalization.py",
        "tests/test_stage90_end_to_end_final_universal_theorem.py",
        "tests/test_proof_driver_status_summary.py",
    ]


def _broad_active_tree_targets():
    return ["tests"]


ENVIRONMENT_SNAPSHOT = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python_executable": sys.executable,
    "python_version": sys.version,
    "platform": platform.platform(),
    "project_root": str(PROJECT_ROOT),
    "artifact_dir": str(ARTIFACT_DIR),
}
ENVIRONMENT_SNAPSHOT


def _json_load(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def _stage_cache_path(name: str) -> Path:
    return STAGE_CACHE_DIR / f"{name}.json"


def _safe_stage_name_component(value) -> str:
    text = str(value).replace("+", "p").replace("-", "m").replace(".", "p")
    return "".join(ch for ch in text if ch.isalnum() or ch == "_")


def _theorem_iv_upper_params():
    return {
        "rho": LIVE_DRIVER_KWARGS.get("rho"),
        "crossing_center": LIVE_DRIVER_KWARGS.get("crossing_center", 0.971635406),
        "atlas_shifts": tuple(LIVE_DRIVER_KWARGS.get("atlas_shifts", (-6.0e-4, -3.0e-4, 0.0, 3.0e-4, 6.0e-4))),
        "n_terms": LIVE_DRIVER_KWARGS.get("n_terms", 10),
        "keep_last": LIVE_DRIVER_KWARGS.get("keep_last", 6),
        "min_q": LIVE_DRIVER_KWARGS.get("min_q", 5),
        "max_q": LIVE_DRIVER_KWARGS.get("max_q"),
        "crossing_half_width": LIVE_DRIVER_KWARGS.get("crossing_half_width", 2.5e-3),
        "band_offset": LIVE_DRIVER_KWARGS.get("band_offset", 5.5e-2),
        "band_width": LIVE_DRIVER_KWARGS.get("band_width", 3.0e-2),
        "target_residue": LIVE_DRIVER_KWARGS.get("target_residue", 0.25),
        "crossing_max_depth": LIVE_DRIVER_KWARGS.get("crossing_max_depth", 5),
        "crossing_min_width": LIVE_DRIVER_KWARGS.get("crossing_min_width", 5e-4),
        "crossing_n_pieces": LIVE_DRIVER_KWARGS.get("crossing_n_pieces", 2),
        "band_initial_subdivisions": LIVE_DRIVER_KWARGS.get("band_initial_subdivisions", 4),
        "band_max_depth": LIVE_DRIVER_KWARGS.get("band_max_depth", 4),
        "band_min_width": LIVE_DRIVER_KWARGS.get("band_min_width", 5e-4),
        "min_tail_members": LIVE_DRIVER_KWARGS.get("min_tail_members", 2),
        "min_cluster_size": LIVE_DRIVER_KWARGS.get("min_cluster_size", 2),
        "min_q_support_fraction": LIVE_DRIVER_KWARGS.get("min_q_support_fraction", 0.6),
        "min_entry_tail_coverage": LIVE_DRIVER_KWARGS.get("min_entry_tail_coverage", 0.75),
        "min_tail_support_fraction": LIVE_DRIVER_KWARGS.get("min_tail_support_fraction", 0.75),
    }


def _build_theorem_iv_upper_tail_coherence_staged(timings):
    params = _theorem_iv_upper_params()
    shift_reports = []
    for shift in params["atlas_shifts"]:
        shift_name = _safe_stage_name_component(f"{shift:.6g}")
        specs = generate_golden_convergent_specs(
            rho=params["rho"],
            n_terms=params["n_terms"],
            keep_last=params["keep_last"],
            min_q=params["min_q"],
            max_q=params["max_q"],
            crossing_center=float(params["crossing_center"]) + float(shift),
            crossing_half_width=params["crossing_half_width"],
            band_offset=params["band_offset"],
            band_width=params["band_width"],
        )
        tail_replacement_min_q = int(params.get("tail_replacement_min_q", 144))
        anchor_specs = [s for s in specs if int(s.q) < tail_replacement_min_q]
        replacement_specs = [s for s in specs if int(s.q) >= tail_replacement_min_q]

        entry_dicts = []
        for spec in anchor_specs:
            spec_name = _safe_stage_name_component(spec.label)
            stage_name = f"theorem_iv_upper_shift_{shift_name}_{spec_name}"
            entry, _ = _build_or_load_stage(
                stage_name,
                lambda spec=spec, entry_dicts=entry_dicts: build_adaptive_incompatibility_entry_certificate(
                    spec.to_approximant_window_spec(),
                    family=FAMILY,
                    target_residue=params["target_residue"],
                    crossing_max_depth=params["crossing_max_depth"],
                    crossing_min_width=params["crossing_min_width"],
                    crossing_n_pieces=params["crossing_n_pieces"],
                    band_initial_subdivisions=params["band_initial_subdivisions"],
                    band_max_depth=params["band_max_depth"],
                    band_min_width=params["band_min_width"],
                    use_seed_propagation=True,
                    previous_entry_dicts=entry_dicts,
                ).to_dict(),
                use_cache=USE_STAGE_CACHE,
                force_rebuild=FORCE_REBUILD_STAGE_CACHE,
                timings=timings,
            )
            entry_dicts.append(entry)

        tail_transport = None
        if replacement_specs:
            replacement_window_specs = [s.to_approximant_window_spec() for s in replacement_specs]
        
            tail_transport, _ = _build_or_load_stage(
                f"theorem_iv_upper_shift_{shift_name}_tail_transport",
                lambda entry_dicts=entry_dicts, replacement_window_specs=replacement_window_specs: build_golden_tail_band_transport_certificate(
                    entry_dicts,
                    replacement_window_specs,
                    family=FAMILY,
                    rho=params["rho"],
                    target_residue=params["target_residue"],
                    explicit_tail_cutoff_q=max(int(e["q"]) for e in entry_dicts) if entry_dicts else int(params["tail_replacement_min_q"]) - 1,
                ).to_dict(),
                use_cache=USE_STAGE_CACHE,
                force_rebuild=FORCE_REBUILD_STAGE_CACHE,
                timings=timings,
            )
        
            # Consume whatever the tail theorem actually produced.
            derived_rows = {}
            if isinstance(tail_transport, dict):
                derived_rows = {int(r["q"]): r for r in tail_transport.get("derived_rows", [])}
        
            derived_specs = []
            missing_specs = []
            for spec in replacement_specs:
                if int(spec.q) in derived_rows:
                    derived_specs.append(spec)
                else:
                    missing_specs.append(spec)
        
            if derived_specs:
                tail_entries = make_tail_transport_entry_dicts(
                    tail_transport,
                    [s.to_approximant_window_spec() for s in derived_specs],
                )
                entry_dicts.extend(tail_entries)
        
            # Rigor-preserving fallback: explicitly build any rows the tail theorem did not derive.
            for spec in missing_specs:
                spec_name = _safe_stage_name_component(spec.label)
                stage_name = f"theorem_iv_upper_shift_{shift_name}_{spec_name}"
                entry, _ = _build_or_load_stage(
                    stage_name,
                    lambda spec=spec: build_adaptive_incompatibility_entry_certificate(
                        spec.to_approximant_window_spec(),
                        family=FAMILY,
                        target_residue=params["target_residue"],
                        crossing_max_depth=params["crossing_max_depth"],
                        crossing_min_width=params["crossing_min_width"],
                        crossing_n_pieces=params["crossing_n_pieces"],
                        band_initial_subdivisions=params["band_initial_subdivisions"],
                        band_max_depth=params["band_max_depth"],
                        band_min_width=params["band_min_width"],
                        use_seed_propagation=True,
                    ).to_dict(),
                    use_cache=USE_STAGE_CACHE,
                    force_rebuild=FORCE_REBUILD_STAGE_CACHE,
                    timings=timings,
                )
                entry_dicts.append(entry)

        shift_report, _ = _build_or_load_stage(
            f"theorem_iv_upper_shift_{shift_name}_report",
            lambda entry_dicts=entry_dicts, shift=shift, specs=specs, tail_transport=tail_transport: build_golden_adaptive_incompatibility_certificate_from_atlas(
                build_adaptive_incompatibility_atlas_certificate_from_entries(
                    entry_dicts, family=FAMILY, min_tail_members=params["min_tail_members"]
                ),
                family=FAMILY,
                rho=float(specs[-1].rho if specs else (params["rho"] if params["rho"] is not None else 0.0)),
                generated_convergents=[s.to_dict() for s in specs],
            ).to_dict(),
            use_cache=USE_STAGE_CACHE,
            force_rebuild=FORCE_REBUILD_STAGE_CACHE,
            timings=timings,
        )
        shift_reports.append({
            "atlas_shift": float(shift),
            "crossing_center": float(params["crossing_center"]) + float(shift),
            "theorem_status": str(shift_report.get("theorem_status", "unknown")),
            "selected_upper_lo": shift_report.get("selected_upper_lo"),
            "selected_upper_hi": shift_report.get("selected_upper_hi"),
            "selected_barrier_lo": shift_report.get("selected_barrier_lo"),
            "selected_barrier_hi": shift_report.get("selected_barrier_hi"),
            "incompatibility_gap": shift_report.get("incompatibility_gap"),
            "witness_qs": [int(x) for x in ((shift_report.get("atlas", {}) or {}).get("hyperbolic_tail", {}) or {}).get("witness_qs", [])],
            "exact_tail_qs": [int(x) for x in ((shift_report.get("atlas", {}) or {}).get("hyperbolic_tail", {}) or {}).get("tail_qs", [])],
            "generated_qs": [int(x) for x in ((shift_report.get("atlas", {}) or {}).get("hyperbolic_tail", {}) or {}).get("generated_qs", [])],
            "tail_transport_certificate": tail_transport,
            "report": shift_report,
        })

    theorem_iv_upper_tail_coherence, _ = _build_or_load_stage(
        "theorem_iv_upper_tail_coherence",
        lambda: build_golden_adaptive_tail_coherence_certificate_from_entries(
            shift_reports,
            family=FAMILY,
            rho=params["rho"],
            crossing_center=params["crossing_center"],
            atlas_shifts=params["atlas_shifts"],
            min_cluster_size=params["min_cluster_size"],
            min_tail_members=params["min_tail_members"],
            min_q_support_fraction=params["min_q_support_fraction"],
            min_entry_tail_coverage=params["min_entry_tail_coverage"],
            min_tail_support_fraction=params["min_tail_support_fraction"],
        ).to_dict(),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    return theorem_iv_upper_tail_coherence

def _build_or_load_stage(name: str, builder, *, use_cache: bool = True, force_rebuild: bool = False, timings: dict | None = None):
    path = _stage_cache_path(name)
    t0 = time.time()
    source = "built"
    print(f"Starting stage: {name}")
    if use_cache and path.exists() and not force_rebuild:
        obj = _json_load(path)
        source = "cache"
    else:
        obj = builder()
        _json_dump(path, obj)
    dt = time.time() - t0
    if timings is not None:
        timings[name] = {
            "source": source,
            "seconds": dt,
            "cache_path": str(path),
        }
    print(f"[{source}] {name}: {dt:.2f}s")
    return obj, path


def _theorem_iv_lower_stage_kwargs():
    return {
        "rho": LIVE_DRIVER_KWARGS.get("rho"),
        "lower_shift_grid": LIVE_DRIVER_KWARGS.get("lower_shift_grid", (-0.015, 0.0, 0.015)),
        "lower_resolution_sets": LIVE_DRIVER_KWARGS.get("lower_resolution_sets", ((64, 96, 128), (80, 112, 144))),
        "sigma_cap": LIVE_DRIVER_KWARGS.get("sigma_cap", 0.04),
        "use_multiresolution": LIVE_DRIVER_KWARGS.get("use_multiresolution", True),
        "oversample_factor": LIVE_DRIVER_KWARGS.get("oversample_factor", 8),
        "lower_min_cluster_size": LIVE_DRIVER_KWARGS.get("lower_min_cluster_size", 2),
    }


def _staged_live_discharge_build():
    timings = {}

    theorem_i_ii, _ = _build_or_load_stage(
        "theorem_i_ii",
        lambda: proof_driver.build_golden_theorem_i_ii_report(family=FAMILY, **LIVE_DRIVER_KWARGS),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iii, _ = _build_or_load_stage(
        "theorem_iii",
        lambda: proof_driver.build_golden_theorem_iii_report(base_K_values=BASE_K_VALUES, family=FAMILY, **LIVE_DRIVER_KWARGS),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv_lower_neighborhood, _ = _build_or_load_stage(
        "theorem_iv_lower_neighborhood",
        lambda: proof_driver.build_golden_theorem_iv_lower_neighborhood_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            **_theorem_iv_lower_stage_kwargs(),
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv_upper_tail_coherence = _build_theorem_iv_upper_tail_coherence_staged(timings)
    theorem_iv_upper_bridge, _ = _build_or_load_stage(
        "theorem_iv_upper_bridge",
        lambda: proof_driver.build_golden_incompatibility_theorem_bridge_report(
            family=FAMILY,
            adaptive_tail_coherence_certificate=theorem_iv_upper_tail_coherence,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv_upper_bridge_promotion, _ = _build_or_load_stage(
        "theorem_iv_upper_bridge_promotion",
        lambda: proof_driver.build_golden_incompatibility_strict_bridge_report(
            family=FAMILY,
            adaptive_tail_coherence_certificate=theorem_iv_upper_tail_coherence,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv_upper_support_core, _ = _build_or_load_stage(
        "theorem_iv_upper_support_core",
        lambda: proof_driver.build_golden_adaptive_support_core_neighborhood_report(
            family=FAMILY,
            baseline_tail_coherence_certificate=theorem_iv_upper_tail_coherence,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv_upper_tail_aware, _ = _build_or_load_stage(
        "theorem_iv_upper_tail_aware",
        lambda: proof_driver.build_golden_adaptive_tail_aware_neighborhood_report(
            family=FAMILY,
            baseline_tail_coherence_certificate=theorem_iv_upper_tail_coherence,
            support_core_neighborhood_certificate=theorem_iv_upper_support_core,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv_upper_tail_stability, _ = _build_or_load_stage(
        "theorem_iv_upper_tail_stability",
        lambda: proof_driver.build_golden_adaptive_tail_stability_report(
            family=FAMILY,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv_upper_bridge_profile, _ = _build_or_load_stage(
        "theorem_iv_upper_bridge_profile",
        lambda: proof_driver.build_golden_incompatibility_bridge_profile_report(
            family=FAMILY,
            adaptive_tail_coherence_certificate=theorem_iv_upper_tail_coherence,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_iv, _ = _build_or_load_stage(
        "theorem_iv",
        lambda: proof_driver.build_golden_theorem_iv_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            lower_neighborhood_stability_certificate=theorem_iv_lower_neighborhood,
            upper_tail_coherence_certificate=theorem_iv_upper_tail_coherence,
            upper_bridge_certificate=theorem_iv_upper_bridge,
            upper_bridge_promotion_certificate=theorem_iv_upper_bridge_promotion,
            upper_support_core_neighborhood_certificate=theorem_iv_upper_support_core,
            upper_tail_aware_neighborhood_certificate=theorem_iv_upper_tail_aware,
            upper_tail_stability_certificate=theorem_iv_upper_tail_stability,
            upper_bridge_profile_certificate=theorem_iv_upper_bridge_profile,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_v_bundle, _ = _build_or_load_stage(
        "theorem_v",
        lambda: proof_driver.build_golden_theorem_v_batched_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            theorem_iii_certificate=theorem_iii,
            theorem_iv_certificate=theorem_iv,
            use_cache=USE_STAGE_CACHE,
            force_rebuild=FORCE_REBUILD_STAGE_CACHE,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    if isinstance(theorem_v_bundle, dict) and 'stage_timings' in theorem_v_bundle:
        for _stage_name, _stage_meta in (theorem_v_bundle.get('stage_timings') or {}).items():
            timings[f"theorem_v::{_stage_name}"] = _stage_meta
    theorem_v_raw = dict((theorem_v_bundle or {}).get("certificate", theorem_v_bundle))
    theorem_v = dict((theorem_v_bundle or {}).get("downstream_certificate") or (theorem_v_bundle or {}).get("compressed_certificate") or theorem_v_raw)
    identification_shell, _ = _build_or_load_stage(
        "identification_shell",
        lambda: proof_driver.build_golden_theorem_ii_to_v_identification_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            theorem_iii_certificate=theorem_iii,
            theorem_iv_certificate=theorem_iv,
            theorem_v_certificate=theorem_v_raw,
            theorem_v_compressed_certificate=theorem_v,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    identification_discharge, _ = _build_or_load_stage(
        "identification_discharge",
        lambda: proof_driver.build_golden_theorem_ii_to_v_identification_discharge_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            theorem_i_ii_certificate=theorem_i_ii,
            theorem_ii_to_v_identification_certificate=identification_shell,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    identification_transport_discharge, _ = _build_or_load_stage(
        "identification_transport_discharge",
        lambda: proof_driver.build_golden_theorem_ii_to_v_identification_transport_discharge_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            theorem_v_certificate=theorem_v_raw,
            threshold_identification_discharge_certificate=identification_discharge,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    identification, _ = _build_or_load_stage(
        "identification_theorem",
        lambda: proof_driver.build_golden_theorem_ii_to_v_identification_theorem_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            threshold_identification_transport_discharge_certificate=identification_transport_discharge,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_vi_base, _ = _build_or_load_stage(
        "theorem_vi_base",
        lambda: proof_driver.build_golden_theorem_vi_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            threshold_identification_certificate=identification,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_vi, _ = _build_or_load_stage(
        "theorem_vi_discharge",
        lambda: proof_driver.build_golden_theorem_vi_discharge_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            theorem_vi_certificate=theorem_vi_base,
            threshold_identification_transport_discharge_certificate=identification_transport_discharge,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_vii_base, _ = _build_or_load_stage(
        "theorem_vii_base",
        lambda: proof_driver.build_golden_theorem_vii_report(
            base_K_values=BASE_K_VALUES,
            challenger_specs=CHALLENGER_SPECS,
            family=FAMILY,
            theorem_vi_envelope_discharge_certificate=theorem_vi,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_vii, _ = _build_or_load_stage(
        "theorem_vii_discharge",
        lambda: proof_driver.build_golden_theorem_vii_discharge_report(
            base_K_values=BASE_K_VALUES,
            challenger_specs=CHALLENGER_SPECS,
            family=FAMILY,
            theorem_vii_certificate=theorem_vii_base,
            theorem_vi_envelope_discharge_certificate=theorem_vi,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_viii_base, _ = _build_or_load_stage(
        "theorem_viii_base",
        lambda: proof_driver.build_golden_theorem_viii_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            theorem_iii_certificate=theorem_iii,
            theorem_iv_certificate=theorem_iv,
            theorem_v_certificate=theorem_v,
            threshold_identification_certificate=identification,
            theorem_vi_certificate=theorem_vi_base,
            theorem_vii_certificate=theorem_vii_base,
            theorem_i_ii_workstream_certificate=theorem_i_ii,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    theorem_viii, _ = _build_or_load_stage(
        "theorem_viii_discharge",
        lambda: proof_driver.build_golden_theorem_viii_discharge_report(
            base_K_values=BASE_K_VALUES,
            family=FAMILY,
            baseline_theorem_viii_certificate=theorem_viii_base,
            theorem_vii_exhaustion_discharge_certificate=theorem_vii,
            theorem_vi_envelope_discharge_certificate=theorem_vi,
            threshold_identification_discharge_certificate=identification_discharge,
            theorem_i_ii_workstream_certificate=theorem_i_ii,
            **LIVE_DRIVER_KWARGS,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )
    live_report, _ = _build_or_load_stage(
        "live_theorem_program_discharge_report_cache",
        lambda: proof_driver.build_golden_theorem_program_status_report_from_subreports(
            theorem_i_ii=theorem_i_ii,
            theorem_iv=theorem_iv,
            theorem_v=theorem_v,
            identification=identification,
            theorem_vi=theorem_vi,
            theorem_vii=theorem_vii,
            theorem_viii=theorem_viii,
            discharge_aware=True,
        ),
        use_cache=USE_STAGE_CACHE,
        force_rebuild=FORCE_REBUILD_STAGE_CACHE,
        timings=timings,
    )

    return live_report, timings


## Live final-discharge run

This notebook now supports two optimized execution modes:

1. the **optimized driver**, which stages shared certificates internally so VI/VII/VIII do not keep rebuilding the same lower/transport/identification objects; and
2. an optional **stage cache** layer, which saves each major theorem object to `artifacts/final_discharge/stage_cache/` so interrupted runs can resume without restarting from scratch.

For a first serious run, keep `USE_STAGE_CACHE = True`.


In [ ]:
live_report = None
live_report_path = None
live_stage_timings = {}

if RUN_LIVE_DISCHARGE:
    if USE_STAGE_CACHE:
        live_report, live_stage_timings = _staged_live_discharge_build()
    else:
        t0 = time.time()
        live_report = proof_driver.build_golden_theorem_program_discharge_status_report(
            base_K_values=BASE_K_VALUES,
            challenger_specs=CHALLENGER_SPECS,
            family=FAMILY,
            **LIVE_DRIVER_KWARGS,
        )
        live_stage_timings = {
            "direct_live_discharge": {
                "source": "built",
                "seconds": time.time() - t0,
                "cache_path": None,
            }
        }
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    live_report_path = _json_dump(
        ARTIFACT_DIR / f"live_theorem_program_discharge_report_{stamp}.json",
        live_report,
    )
    print("Live discharge report built successfully.")
    _display_records(
        [
            {"stage": name, **meta}
            for name, meta in sorted(live_stage_timings.items(), key=lambda kv: kv[1]["seconds"], reverse=True)
        ],
        title="Live-discharge stage timings",
    )
else:
    print("RUN_LIVE_DISCHARGE is False; skipping live theorem-program discharge build.")


Starting stage: theorem_i_ii
[cache] theorem_i_ii: 0.02s
Starting stage: theorem_iii
[cache] theorem_iii: 0.01s
Starting stage: theorem_iv_lower_neighborhood
[cache] theorem_iv_lower_neighborhood: 0.01s
Starting stage: theorem_iv_upper_shift_m0p0006_goldm1321
[cache] theorem_iv_upper_shift_m0p0006_goldm1321: 0.00s
Starting stage: theorem_iv_upper_shift_m0p0006_goldm2134
[cache] theorem_iv_upper_shift_m0p0006_goldm2134: 0.00s
Starting stage: theorem_iv_upper_shift_m0p0006_goldm3455
[cache] theorem_iv_upper_shift_m0p0006_goldm3455: 0.00s
Starting stage: theorem_iv_upper_shift_m0p0006_goldm5589
[cache] theorem_iv_upper_shift_m0p0006_goldm5589: 0.00s
Starting stage: theorem_iv_upper_shift_m0p0006_tail_transport
[cache] theorem_iv_upper_shift_m0p0006_tail_transport: 0.00s
Starting stage: theorem_iv_upper_shift_m0p0006_report
[cache] theorem_iv_upper_shift_m0p0006_report: 0.01s
Starting stage: theorem_iv_upper_shift_m0p0003_goldm1321
[cache] theorem_iv_upper_shift_m0p0003_goldm1321: 0.00s
St

In [ ]:
if live_report is None:
    raise RuntimeError("No live report is available. Set RUN_LIVE_DISCHARGE = True and rerun the previous cell.")

implementation_summary = dict(live_report.get("implementation_summary", {}))
acceptance_snapshot = _build_acceptance_snapshot(live_report)
acceptance_checks = _acceptance_checks(acceptance_snapshot)
next_step_summary = _summarize_next_step(acceptance_snapshot)

print("Implementation summary:")
pprint(implementation_summary)

print("\nAcceptance snapshot:")
pprint(acceptance_snapshot)

print("\nNotebook diagnosis:")
pprint(next_step_summary)

_display_records(acceptance_checks, title="Acceptance checks")

In [ ]:
theorem_rows = _theorem_rows_table(live_report)
_display_records(theorem_rows, title="Theorem status rows")

## Hard acceptance gate (optional)

Turn `FAIL_HARD_ON_ACCEPTANCE = True` in the configuration cell if you want this notebook to stop as soon as the live route fails the closure target.

By default, the cell below **reports** failures but does not interrupt the rest of the notebook, since the main value of this notebook is often the diagnosis when closure has not yet been reached.

In [ ]:
failed_checks = [row for row in acceptance_checks if not row["passed"]]
print("Failed acceptance checks:", len(failed_checks))
for row in failed_checks:
    print("-", row["check"], "->", row["value"])

if FAIL_HARD_ON_ACCEPTANCE and failed_checks:
    raise AssertionError("Live discharge did not satisfy all acceptance checks.")

## Deep dive: Theorem VIII burdens and current reduction geometry

Theorem VIII is the canonical endgame consumer. If the live run is not fully closed, the fields below are the ones that should drive the next move.

In [ ]:
theorem_viii = (live_report.get("subreports") or {}).get("theorem_viii", {})
geometry = live_report.get("current_reduction_geometry_summary", {})

viii_focus = {
    "theorem_status": theorem_viii.get("theorem_status"),
    "statement_mode": theorem_viii.get("statement_mode"),
    "final_certificate_ready_for_code_path": theorem_viii.get("final_certificate_ready_for_code_path"),
    "final_certificate_ready_for_paper": theorem_viii.get("final_certificate_ready_for_paper"),
    "remaining_true_mathematical_burden": theorem_viii.get("remaining_true_mathematical_burden", []),
    "remaining_workstream_paper_grade_burden": theorem_viii.get("remaining_workstream_paper_grade_burden", []),
    "remaining_exhaustion_paper_grade_burden": theorem_viii.get("remaining_exhaustion_paper_grade_burden", []),
    "current_reduction_geometry_summary": geometry,
}

pprint(viii_focus)

In [ ]:
subreports = live_report.get("subreports", {})
subreport_focus = []
for key, value in subreports.items():
    subreport_focus.append({
        "key": key,
        "theorem_status": value.get("theorem_status"),
        "statement_mode": value.get("statement_mode"),
        "open_hypotheses": list(value.get("open_hypotheses", []) or []),
        "active_assumptions": list(value.get("active_assumptions", []) or []),
    })
_display_records(subreport_focus, title="Subreport focus")

## Regression tiers

These tiers are intentionally separate.

### Tier 1: stage-84 through stage-90 theorem-program tests
This covers the late theorem-program buildout.

### Tier 2: focused endgame/driver slice
This includes the proof-driver and final-reduction tests that are most directly relevant to the live-discharge story.

### Tier 3: broad active-tree run
This is the broader `pytest tests` pass with `stage35/` explicitly excluded from discovery.

> Recommendation: run the live discharge first, then tier 1, then tier 2, then tier 3.

In [ ]:
stage84_to_stage90_tests = _collect_stage84_to_stage90_tests()
focused_endgame_tests = _focused_endgame_tests()
broad_active_tree_targets = _broad_active_tree_targets()

print("Tier 1 test count:", len(stage84_to_stage90_tests))
print("Tier 2 test count:", len(focused_endgame_tests))
print("Tier 3 targets:", broad_active_tree_targets)

In [ ]:
regression_results = {}

if RUN_REGRESSION_TIER_1:
    regression_results["tier_1_stage84_to_stage90"] = _run_command(_pytest_cmd(stage84_to_stage90_tests))
else:
    print("RUN_REGRESSION_TIER_1 is False; skipping tier 1.")

if RUN_REGRESSION_TIER_2:
    regression_results["tier_2_focused_endgame"] = _run_command(_pytest_cmd(focused_endgame_tests))
else:
    print("RUN_REGRESSION_TIER_2 is False; skipping tier 2.")

if RUN_REGRESSION_TIER_3:
    regression_results["tier_3_broad_active_tree"] = _run_command(_pytest_cmd(broad_active_tree_targets))
else:
    print("RUN_REGRESSION_TIER_3 is False; skipping tier 3.")

In [ ]:
if regression_results:
    regression_summary = []
    for key, result in regression_results.items():
        regression_summary.append({
            "tier": key,
            "returncode": result["returncode"],
            "elapsed_seconds": round(result["elapsed_seconds"], 2),
            "passed": result["returncode"] == 0,
        })
    _display_records(regression_summary, title="Regression summary")
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    _json_dump(ARTIFACT_DIR / f"regression_summary_{stamp}.json", regression_results)
else:
    print("No regression tiers were run in this notebook execution.")

## Interpretation guide

Use the live discharge result first, then interpret the regression tiers.

### Best-case outcome
If all of the following hold:
- `final_universal_theorem_ready_for_code_path == True`
- `final_universal_theorem_ready_for_paper == True`
- `remaining_true_mathematical_burden == []`
- `paper_grade_burden_remaining == []`
- `current_reduction_geometry_status == "current-reduction-geometry-strong"`

then the project is signaling **full live closure** on the canonical default route.

### Mathematics closed, packaging still open
If:
- code-path is ready,
- true mathematical burden is empty,
- but paper-grade readiness is still false,

then the next move is not a new theorem front. It is likely Workstream-A / Theorem VII / citation or finality cleanup.

### Live mathematical burden still present
If `remaining_true_mathematical_burden` is nonempty, that list should be treated as the canonical next-step oracle. Do not guess the next move from the earlier staged tests if the live VIII consumer is already telling you what remains.

### Regression interpretation caution
Passing the stage-89/stage-90 reduction tests is valuable, but they still include staged constructions of final objects. The **live discharge cell above** is the stronger closure signal.

In [ ]:
run_manifest = {
    "environment": ENVIRONMENT_SNAPSHOT,
    "base_K_values": BASE_K_VALUES,
    "live_driver_kwargs": LIVE_DRIVER_KWARGS,
    "ran_live_discharge": RUN_LIVE_DISCHARGE,
    "live_report_path": None if live_report_path is None else str(live_report_path),
    "acceptance_snapshot": acceptance_snapshot,
    "acceptance_checks": acceptance_checks,
    "next_step_summary": next_step_summary,
    "regression_tiers_enabled": {
        "tier_1": RUN_REGRESSION_TIER_1,
        "tier_2": RUN_REGRESSION_TIER_2,
        "tier_3": RUN_REGRESSION_TIER_3,
    },
}
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
_json_dump(ARTIFACT_DIR / f"notebook_run_manifest_{stamp}.json", run_manifest)
run_manifest